# MDSP California Housing Analysis

## Objectives

This notebook performs a complete, reproducible, privacy-conscious analysis of a local California housing dataset. It addresses five research questions concerning income, geography, density and crowding proxies, housing-stock age, and predictive performance.

Only aggregate statistics, aggregate diagnostics, model summaries, and privacy-conscious figures are displayed or exported. Raw records, exact coordinate pairs, row-level predictions, residuals, fold assignments, and transformed observations are not displayed or saved.

## Privacy, observational design, and ecological inference

- The source data remain local.
- Each row is treated as an **area-level observation**, not an individual property, person, or household.
- Findings are descriptive associations or predictive relationships, not causal effects.
- Area-level results must not be transferred automatically to individuals or households; doing so risks ecological fallacy.
- Spatial dependence, residual confounding, measurement error, and target top-coding may affect results.
- The controls in this notebook reduce unnecessary disclosure risk but do not guarantee anonymity or regulatory compliance.

## 1. Reproducibility settings and package checks

In [ ]:
from pathlib import Path
import os
import sys
import platform
import importlib
import importlib.metadata
import warnings

RANDOM_SEED = 42
FAST_MODE = True
SAVE_FIGURES = True

RUN_DIRECTORY = Path.cwd()
OUTPUT_DIRECTORY = RUN_DIRECTORY / "mdsp_outputs"
FIGURE_DIRECTORY = OUTPUT_DIRECTORY / "figures"
TABLE_DIRECTORY = OUTPUT_DIRECTORY / "tables"

for directory in (OUTPUT_DIRECTORY, FIGURE_DIRECTORY, TABLE_DIRECTORY):
    directory.mkdir(parents=True, exist_ok=True)

os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
warnings.filterwarnings("default")

CORE_PACKAGES = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
}

missing = []
versions = {}
for import_name, distribution_name in CORE_PACKAGES.items():
    try:
        importlib.import_module(import_name)
        versions[distribution_name] = importlib.metadata.version(distribution_name)
    except (ImportError, importlib.metadata.PackageNotFoundError):
        missing.append(distribution_name)

if missing:
    print("Missing required packages:", ", ".join(missing))
    print("# Optional manual installation command:")
    print("# python -m pip install " + " ".join(missing))
    raise ImportError("Install the missing core packages and restart the kernel.")

try:
    import statsmodels
    STATSMODELS_AVAILABLE = True
    versions["statsmodels"] = importlib.metadata.version("statsmodels")
except (ImportError, importlib.metadata.PackageNotFoundError):
    STATSMODELS_AVAILABLE = False
    versions["statsmodels"] = "not installed; fallback methods enabled"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display, Markdown

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score,
)
from sklearn.model_selection import GroupKFold, KFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures

np.random.seed(RANDOM_SEED)

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "figure.constrained_layout.use": True,
})

display(pd.DataFrame({"package": list(versions), "version": list(versions.values())}))
print(f"Python: {platform.python_version()}")
print(f"Platform: {platform.platform()}")
print(f"Random seed: {RANDOM_SEED}")
print(f"Fast mode: {FAST_MODE}")

if not STATSMODELS_AVAILABLE:
    display(Markdown(
        "**Optional dependency:** `statsmodels` is unavailable. Bootstrap or "
        "SciPy fallbacks will be used. Optional manual command: "
        "`python -m pip install statsmodels`."
    ))

## 2. Safe local data acquisition

The notebook first uses an existing pandas DataFrame named `mydata`. If it is absent, set `LOCAL_CSV_PATH` to a local CSV file. Network access is not used.

In [ ]:
LOCAL_CSV_PATH = Path("")  # Example: Path(r"C:\data\housing.csv")

EXPECTED_COLUMNS = [
    "longitude",
    "latitude",
    "housing_median_age",
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
    "median_income",
    "median_house_value",
]

def load_local_csv(path):
    # Load strictly from a local CSV without displaying its path or records.
    if not str(path):
        raise FileNotFoundError(
            "Define `mydata` before this cell or set LOCAL_CSV_PATH to a local CSV."
        )
    resolved = path.expanduser().resolve()
    if not resolved.exists() or not resolved.is_file():
        raise FileNotFoundError("The configured local CSV file is unavailable.")
    if resolved.suffix.lower() != ".csv":
        raise ValueError("The configured local file must have a .csv extension.")
    if resolved.stat().st_size == 0:
        raise ValueError("The configured CSV is empty.")

    last_decode_error = None
    for encoding in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
        try:
            return pd.read_csv(
                resolved,
                encoding=encoding,
                engine="python",
                on_bad_lines="error",
            )
        except UnicodeDecodeError as exc:
            last_decode_error = exc
        except pd.errors.EmptyDataError as exc:
            raise ValueError("The configured CSV contains no parsable data.") from exc
        except pd.errors.ParserError as exc:
            raise ValueError("The configured CSV is malformed.") from exc
    raise UnicodeError("The CSV could not be decoded safely.") from last_decode_error

if "mydata" in globals():
    if not isinstance(mydata, pd.DataFrame):
        raise TypeError("The existing object `mydata` is not a pandas DataFrame.")
    source_mode = "existing_dataframe"
else:
    mydata = load_local_csv(LOCAL_CSV_PATH)
    source_mode = "local_csv"

if mydata.empty:
    raise ValueError("The dataset must contain at least one row and one column.")

analysis_data = mydata.copy(deep=True)

print("Local dataset acquired successfully.")
print(f"Source mode: {source_mode}")
print(f"Dimensions: {analysis_data.shape[0]:,} rows × {analysis_data.shape[1]} columns")

**Interpretation:** Only dimensions and a sanitized source mode are shown. No file path, raw row, or coordinate pair is displayed.

## 3. Schema, dimensions, and data-type validation

In [ ]:
missing_required = [c for c in EXPECTED_COLUMNS if c not in analysis_data.columns]
if missing_required:
    raise KeyError("Missing required columns: " + ", ".join(missing_required))

schema_summary = pd.DataFrame({
    "column": EXPECTED_COLUMNS,
    "dtype": [str(analysis_data[c].dtype) for c in EXPECTED_COLUMNS],
    "non_null_count": [int(analysis_data[c].notna().sum()) for c in EXPECTED_COLUMNS],
    "missing_count": [int(analysis_data[c].isna().sum()) for c in EXPECTED_COLUMNS],
    "numeric_dtype": [
        bool(pd.api.types.is_numeric_dtype(analysis_data[c]))
        for c in EXPECTED_COLUMNS
    ],
})
display(schema_summary)

non_numeric = schema_summary.loc[~schema_summary["numeric_dtype"], "column"].tolist()
if non_numeric:
    raise TypeError("Required numeric columns have nonnumeric dtypes: " + ", ".join(non_numeric))

dimension_summary = pd.DataFrame([{
    "observed_rows": int(analysis_data.shape[0]),
    "expected_rows_approximate": 20000,
    "observed_columns": int(analysis_data.shape[1]),
    "required_columns": len(EXPECTED_COLUMNS),
    "unexpected_columns": int(len([c for c in analysis_data.columns if c not in EXPECTED_COLUMNS])),
}])
display(dimension_summary)

if len(analysis_data) != 20000:
    display(Markdown(
        f"**Dimension note:** The observed row count is {len(analysis_data):,}. "
        "The notebook continues because 20,000 is treated as approximate."
    ))

## 4. Data-quality, missingness, invalid values, and outlier diagnostics

In [ ]:
def dominant_percentage(series):
    frequencies = series.value_counts(dropna=False, normalize=True)
    return 100 * float(frequencies.iloc[0]) if not frequencies.empty else np.nan

def iqr_flag_count(series):
    values = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    if values.empty:
        return 0
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    if iqr == 0:
        return 0
    return int(((values < q1 - 1.5 * iqr) | (values > q3 + 1.5 * iqr)).sum())

def robust_z_flag_count(series):
    values = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    if values.empty:
        return 0
    median = float(values.median())
    mad = float(np.median(np.abs(values - median)))
    if mad == 0:
        return 0
    robust_z = 0.6745 * (values - median) / mad
    return int((np.abs(robust_z) > 3.5).sum())

quality_rows = []
for column in EXPECTED_COLUMNS:
    series = analysis_data[column]
    dominant = dominant_percentage(series)
    unique = int(series.nunique(dropna=True))
    quality_rows.append({
        "column": column,
        "missing_count": int(series.isna().sum()),
        "missing_percentage": round(100 * series.isna().mean(), 3),
        "infinite_count": int(np.isinf(series.to_numpy(dtype=float, na_value=np.nan)).sum()),
        "unique_count": unique,
        "constant": unique <= 1,
        "near_constant_95pct": bool(unique > 1 and dominant >= 95),
        "dominant_value_percentage": round(dominant, 2),
        "skewness": round(float(series.skew()), 3),
        "iqr_flag_count": iqr_flag_count(series),
        "robust_z_flag_count": robust_z_flag_count(series),
    })

data_quality_summary = pd.DataFrame(quality_rows)
display(data_quality_summary)

duplicate_count = int(analysis_data.duplicated().sum())
duplicate_percentage = 100 * duplicate_count / len(analysis_data)
print(f"Duplicate rows: {duplicate_count:,} ({duplicate_percentage:.2f}%)")

rules = {
    "longitude outside [-180, 180]": ~analysis_data["longitude"].between(-180, 180) & analysis_data["longitude"].notna(),
    "latitude outside [-90, 90]": ~analysis_data["latitude"].between(-90, 90) & analysis_data["latitude"].notna(),
    "negative housing age": analysis_data["housing_median_age"] < 0,
    "negative total rooms": analysis_data["total_rooms"] < 0,
    "negative total bedrooms": analysis_data["total_bedrooms"] < 0,
    "bedrooms greater than rooms": analysis_data["total_bedrooms"] > analysis_data["total_rooms"],
    "negative population": analysis_data["population"] < 0,
    "negative households": analysis_data["households"] < 0,
    "zero households denominator": analysis_data["households"] == 0,
    "zero rooms denominator": analysis_data["total_rooms"] == 0,
    "negative median income": analysis_data["median_income"] < 0,
    "negative recorded house value": analysis_data["median_house_value"] < 0,
}

invalid_summary = pd.DataFrame([
    {
        "rule": name,
        "affected_count": int(mask.fillna(False).sum()),
        "affected_percentage": round(100 * mask.fillna(False).mean(), 3),
    }
    for name, mask in rules.items()
])
display(invalid_summary)

**Interpretation:** IQR and robust-z flags identify unusual observations rather than confirmed errors. No row is removed, capped, or modified automatically.

## 5. Privacy-conscious distribution and multicollinearity assessment

In [ ]:
generated_files = []

def save_figure(fig, stem):
    if not SAVE_FIGURES:
        return
    png = FIGURE_DIRECTORY / f"{stem}.png"
    svg = FIGURE_DIRECTORY / f"{stem}.svg"
    fig.savefig(png, dpi=300, bbox_inches="tight")
    generated_files.append(png)
    try:
        fig.savefig(svg, bbox_inches="tight")
        generated_files.append(svg)
    except (OSError, ValueError):
        print(f"SVG export was unavailable for {stem}; PNG was retained.")

plot_columns = [
    "housing_median_age",
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
    "median_income",
    "median_house_value",
]

fig, axes = plt.subplots(4, 2, figsize=(10, 12))
axes = axes.ravel()
for ax, column in zip(axes, plot_columns):
    values = analysis_data[column].replace([np.inf, -np.inf], np.nan).dropna()
    ax.hist(values, bins=40, alpha=0.85)
    ax.set_title(column.replace("_", " ").title())
    ax.set_xlabel("Observed value")
    ax.set_ylabel("Area-level observation count")
    ax.grid(alpha=0.2)
for ax in axes[len(plot_columns):]:
    ax.set_visible(False)
fig.suptitle("Aggregate distributions of non-coordinate variables")
save_figure(fig, "01_non_coordinate_distributions")
plt.show()

spearman_matrix = analysis_data[EXPECTED_COLUMNS].corr(method="spearman")
fig, ax = plt.subplots(figsize=(9, 7))
image = ax.imshow(spearman_matrix, cmap="viridis", vmin=-1, vmax=1)
ax.set_xticks(range(len(EXPECTED_COLUMNS)), [c.replace("_", "\n") for c in EXPECTED_COLUMNS], rotation=45, ha="right")
ax.set_yticks(range(len(EXPECTED_COLUMNS)), [c.replace("_", " ") for c in EXPECTED_COLUMNS])
ax.set_title("Spearman correlation matrix")
fig.colorbar(image, ax=ax, label="Spearman correlation")
save_figure(fig, "02_spearman_correlation")
plt.show()

condition_number = float(np.linalg.cond(spearman_matrix.fillna(0).to_numpy()))
print(f"Correlation-matrix condition number: {condition_number:.2f}")

**Interpretation:** Distribution shape and aggregate correlation inform robust modeling and multicollinearity assessment. Correlation is not causation.

## 6. Target top-coding investigation

In [ ]:
target = analysis_data["median_house_value"].replace([np.inf, -np.inf], np.nan).dropna()
target_maximum = float(target.max())
count_at_maximum = int(np.isclose(target, target_maximum, rtol=0, atol=1e-9).sum())
count_near_500001 = int(np.isclose(target, 500001, rtol=0, atol=1).sum())
percent_at_maximum = 100 * count_at_maximum / len(target)
percent_near_500001 = 100 * count_near_500001 / len(target)

top_coding_detected = bool(
    (count_near_500001 > 0 and percent_near_500001 >= 0.5)
    or percent_at_maximum >= 0.5
)

topcoding_summary = pd.DataFrame([{
    "maximum_recorded_target": round(target_maximum, 2),
    "count_at_maximum": count_at_maximum,
    "percentage_at_maximum": round(percent_at_maximum, 3),
    "count_at_or_near_500001": count_near_500001,
    "percentage_at_or_near_500001": round(percent_near_500001, 3),
    "top_coding_signal_detected": top_coding_detected,
}])
display(topcoding_summary)

upper_tail = target[target >= target.quantile(0.90)]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(upper_tail, bins=35, alpha=0.85)
ax.set_title("Upper 10% of recorded median housing values")
ax.set_xlabel("Recorded median housing value")
ax.set_ylabel("Area-level observation count")
ax.grid(alpha=0.2)
save_figure(fig, "03_target_upper_tail")
plt.show()

if top_coding_detected:
    display(Markdown(
        f"**Assessment:** {percent_at_maximum:.2f}% of non-missing observations "
        "equal the recorded maximum. Models therefore predict recorded, potentially "
        "capped values rather than uncensored market values."
    ))
else:
    display(Markdown(
        "**Assessment:** The implemented concentration rule did not detect a strong "
        "top-coding signal. This is not proof that censoring is absent."
    ))

## 7. Privacy-conscious exploratory visualization

In [ ]:
def quantile_bin(series, bins):
    ranked = series.rank(method="first")
    return pd.qcut(ranked, q=bins, labels=False, duplicates="drop")

longitude_bin = quantile_bin(analysis_data["longitude"], 8)
latitude_bin = quantile_bin(analysis_data["latitude"], 8)

geo_summary = (
    pd.DataFrame({
        "longitude_bin": longitude_bin,
        "latitude_bin": latitude_bin,
        "target": analysis_data["median_house_value"],
    })
    .dropna()
    .groupby(["latitude_bin", "longitude_bin"], observed=True)
    .agg(count=("target", "size"), median_value=("target", "median"))
    .reset_index()
)
geo_summary.loc[geo_summary["count"] < 5, "median_value"] = np.nan

geo_matrix = geo_summary.pivot(
    index="latitude_bin",
    columns="longitude_bin",
    values="median_value",
)

fig, ax = plt.subplots(figsize=(8, 5.5))
image = ax.imshow(geo_matrix.sort_index(ascending=False), cmap="viridis", aspect="auto")
ax.set_title("Recorded housing value by generalized spatial bin")
ax.set_xlabel("Longitude quantile-bin index")
ax.set_ylabel("Latitude quantile-bin index")
fig.colorbar(image, ax=ax, label="Median recorded housing value")
save_figure(fig, "04_generalized_spatial_value")
plt.show()

print(f"Suppressed spatial bins with fewer than five observations: {(geo_summary['count'] < 5).sum()}")

fig, ax = plt.subplots(figsize=(8, 5))
hexbin = ax.hexbin(
    analysis_data["median_income"],
    analysis_data["median_house_value"],
    gridsize=45,
    mincnt=5,
    cmap="viridis",
)
ax.set_title("Median income and recorded housing value")
ax.set_xlabel("Median income")
ax.set_ylabel("Recorded median housing value")
fig.colorbar(hexbin, ax=ax, label="Observation count per hexagon")
save_figure(fig, "05_income_value_hexbin")
plt.show()

**Interpretation:** Geographic results are displayed through generalized bin indices. Sparse bins are suppressed, and no exact longitude–latitude pairs are shown.

## 8. Feature engineering and invalid denominators

In [ ]:
engineered_data = analysis_data.copy(deep=True)

ratio_specs = {
    "population_per_household": ("population", "households"),
    "rooms_per_household": ("total_rooms", "households"),
    "bedrooms_per_household": ("total_bedrooms", "households"),
    "bedroom_to_room_ratio": ("total_bedrooms", "total_rooms"),
}

engineering_log = []
for new_feature, (numerator, denominator) in ratio_specs.items():
    invalid_denominator = (
        engineered_data[denominator].isna()
        | ~np.isfinite(engineered_data[denominator])
        | (engineered_data[denominator] <= 0)
    )
    engineered_data[new_feature] = np.nan
    valid = ~invalid_denominator
    engineered_data.loc[valid, new_feature] = (
        engineered_data.loc[valid, numerator]
        / engineered_data.loc[valid, denominator]
    )
    engineered_data[new_feature] = engineered_data[new_feature].replace([np.inf, -np.inf], np.nan)

    engineering_log.append({
        "engineered_feature": new_feature,
        "invalid_denominator_count": int(invalid_denominator.sum()),
        "resulting_missing_count": int(engineered_data[new_feature].isna().sum()),
        "resulting_missing_percentage": round(100 * engineered_data[new_feature].isna().mean(), 3),
        "skewness": round(float(engineered_data[new_feature].skew()), 3),
    })

feature_engineering_summary = pd.DataFrame(engineering_log)
display(feature_engineering_summary)

**Interpretation:** Invalid divisions are converted to missing values and handled within training-fold pipelines. The engineered variables are area-level proxies, not direct measurements of individual crowding.

## 9. Spatial-block construction and validation protocol

In [ ]:
def construct_spatial_groups(longitude, latitude, bins_per_axis=5, minimum_size=5):
    lon_bin = quantile_bin(longitude, bins_per_axis)
    lat_bin = quantile_bin(latitude, bins_per_axis)
    groups = lon_bin.astype("Int64").astype(str) + "_" + lat_bin.astype("Int64").astype(str)
    counts = groups.value_counts()
    sparse = counts[counts < minimum_size].index
    groups = groups.where(~groups.isin(sparse), "sparse_merged")
    final_counts = groups.value_counts()
    summary = pd.DataFrame([{
        "spatial_group_count": int(final_counts.size),
        "minimum_group_size": int(final_counts.min()),
        "median_group_size": float(final_counts.median()),
        "maximum_group_size": int(final_counts.max()),
        "groups_below_five_after_merge": int((final_counts < minimum_size).sum()),
    }])
    return groups.astype(str), summary

spatial_groups, spatial_group_summary = construct_spatial_groups(
    engineered_data["longitude"],
    engineered_data["latitude"],
)
display(spatial_group_summary)

SPATIAL_OUTER_FOLDS = min(5, int(spatial_groups.nunique()))
SPATIAL_INNER_FOLDS = 3
RANDOM_OUTER_FOLDS = 5
RANDOM_INNER_FOLDS = 3

if SPATIAL_OUTER_FOLDS < 3:
    raise ValueError("At least three spatial groups are required.")

print(f"Primary spatial outer folds: {SPATIAL_OUTER_FOLDS}")
print(f"Spatial inner folds: {SPATIAL_INNER_FOLDS}")
print(f"Secondary random outer folds: {RANDOM_OUTER_FOLDS}")

**Interpretation:** Exact coordinate-to-group assignments remain internal. Spatial validation is primary; random validation is secondary because geographically mixed folds may overestimate generalization.

## 10. Leakage-safe preprocessing and evaluation utilities

In [ ]:
TARGET = "median_house_value"
ENGINEERED_FEATURES = list(ratio_specs)
MODEL_FEATURES = [
    "longitude",
    "latitude",
    "housing_median_age",
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
    "median_income",
    *ENGINEERED_FEATURES,
]

model_frame = engineered_data[MODEL_FEATURES + [TARGET]].copy()
valid_target = model_frame[TARGET].notna() & np.isfinite(model_frame[TARGET])
model_frame = model_frame.loc[valid_target]
model_groups = spatial_groups.loc[valid_target]
X_all = model_frame[MODEL_FEATURES]
y_all = model_frame[TARGET]

excluded_target_rows = int((~valid_target).sum())
print(f"Rows excluded from modeling for missing/infinite target: {excluded_target_rows:,}")
print(f"Rows used for modeling: {len(model_frame):,}")

SPATIAL_SPLITS = list(
    GroupKFold(n_splits=SPATIAL_OUTER_FOLDS).split(X_all, y_all, groups=model_groups)
)
RANDOM_SPLITS = list(
    KFold(
        n_splits=RANDOM_OUTER_FOLDS,
        shuffle=True,
        random_state=RANDOM_SEED,
    ).split(X_all, y_all)
)

def standard_pipeline(model):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", model),
    ])

def metrics(y_true, predictions):
    return {
        "mae": mean_absolute_error(y_true, predictions),
        "rmse": mean_squared_error(y_true, predictions) ** 0.5,
        "r2": r2_score(y_true, predictions),
        "median_ae": median_absolute_error(y_true, predictions),
    }

def summarize_folds(frame):
    result = {}
    n = len(frame)
    t_value = stats.t.ppf(0.975, max(n - 1, 1))
    for metric in ("mae", "rmse", "r2", "median_ae"):
        values = frame[metric].astype(float)
        mean = float(values.mean())
        std = float(values.std(ddof=1)) if n > 1 else 0.0
        half = t_value * std / np.sqrt(n) if n > 1 else 0.0
        result[f"{metric}_mean"] = mean
        result[f"{metric}_std"] = std
        result[f"{metric}_ci_low"] = mean - half
        result[f"{metric}_ci_high"] = mean + half
    return result

def nested_evaluate(name, pipeline, grid, X, y, outer_splits, strategy, groups=None):
    records = []
    selected = []
    for fold_number, (train_idx, test_idx) in enumerate(outer_splits, start=1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        if grid:
            if strategy == "spatial":
                train_groups = groups.iloc[train_idx]
                inner_folds = min(SPATIAL_INNER_FOLDS, int(train_groups.nunique()))
                if inner_folds < 2:
                    raise ValueError("Insufficient spatial groups for nested tuning.")
                inner_cv = GroupKFold(n_splits=inner_folds)
                search = GridSearchCV(
                    clone(pipeline),
                    grid,
                    scoring="neg_mean_absolute_error",
                    cv=inner_cv,
                    n_jobs=-1,
                    error_score="raise",
                )
                search.fit(X_train, y_train, groups=train_groups)
            else:
                inner_cv = KFold(
                    n_splits=RANDOM_INNER_FOLDS,
                    shuffle=True,
                    random_state=RANDOM_SEED + fold_number,
                )
                search = GridSearchCV(
                    clone(pipeline),
                    grid,
                    scoring="neg_mean_absolute_error",
                    cv=inner_cv,
                    n_jobs=-1,
                    error_score="raise",
                )
                search.fit(X_train, y_train)
            fitted = search.best_estimator_
            selected.append(search.best_params_)
        else:
            fitted = clone(pipeline).fit(X_train, y_train)
            selected.append({})

        prediction = fitted.predict(X_test)
        records.append({
            "model": name,
            "validation_strategy": strategy,
            "outer_fold": fold_number,
            **metrics(y_test, prediction),
        })

    fold_frame = pd.DataFrame(records)
    aggregate = {
        "model": name,
        "validation_strategy": strategy,
        "outer_folds": len(fold_frame),
        **summarize_folds(fold_frame),
    }
    return fold_frame, aggregate, selected

def fixed_evaluate(name, pipeline, X, y, splits, strategy):
    records = []
    for fold_number, (train_idx, test_idx) in enumerate(splits, start=1):
        fitted = clone(pipeline).fit(X.iloc[train_idx], y.iloc[train_idx])
        prediction = fitted.predict(X.iloc[test_idx])
        records.append({
            "model": name,
            "validation_strategy": strategy,
            "outer_fold": fold_number,
            **metrics(y.iloc[test_idx], prediction),
        })
    frame = pd.DataFrame(records)
    return frame, {
        "model": name,
        "validation_strategy": strategy,
        "outer_folds": len(frame),
        **summarize_folds(frame),
    }

## 11. Research Question 1 — Income and recorded housing value

In [ ]:
rq1 = engineered_data[["median_income", TARGET]].replace([np.inf, -np.inf], np.nan).dropna()

pearson = stats.pearsonr(rq1["median_income"], rq1[TARGET])
spearman = stats.spearmanr(rq1["median_income"], rq1[TARGET])

correlation_results = pd.DataFrame([
    {"method": "Pearson", "estimate": pearson.statistic, "p_value": pearson.pvalue},
    {"method": "Spearman", "estimate": spearman.statistic, "p_value": spearman.pvalue},
])
display(correlation_results.round(5))

if STATSMODELS_AVAILABLE:
    import statsmodels.api as sm
    rq1_design = sm.add_constant(rq1[["median_income"]])
    rq1_fit = sm.OLS(rq1[TARGET], rq1_design).fit(cov_type="HC3")
    ci = rq1_fit.conf_int().loc["median_income"]
    rq1_linear_result = pd.DataFrame([{
        "method": "OLS with HC3 standard errors",
        "income_coefficient": rq1_fit.params["median_income"],
        "ci_95_low": ci.iloc[0],
        "ci_95_high": ci.iloc[1],
        "p_value": rq1_fit.pvalues["median_income"],
        "r_squared": rq1_fit.rsquared,
        "n": int(rq1_fit.nobs),
    }])
else:
    regression = stats.linregress(rq1["median_income"], rq1[TARGET])
    rq1_linear_result = pd.DataFrame([{
        "method": "SciPy linear regression fallback",
        "income_coefficient": regression.slope,
        "ci_95_low": regression.slope - 1.96 * regression.stderr,
        "ci_95_high": regression.slope + 1.96 * regression.stderr,
        "p_value": regression.pvalue,
        "r_squared": regression.rvalue ** 2,
        "n": len(rq1),
    }])
display(rq1_linear_result.round(4))

rq1_groups = spatial_groups.loc[rq1.index]
rq1_splits = list(
    GroupKFold(n_splits=min(5, int(rq1_groups.nunique()))).split(
        rq1[["median_income"]], rq1[TARGET], groups=rq1_groups
    )
)

linear_income = standard_pipeline(Ridge(alpha=10))
quadratic_income = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=10)),
])

_, linear_summary = fixed_evaluate(
    "Linear income specification",
    linear_income,
    rq1[["median_income"]],
    rq1[TARGET],
    rq1_splits,
    "spatial",
)
_, quadratic_summary = fixed_evaluate(
    "Quadratic income specification",
    quadratic_income,
    rq1[["median_income"]],
    rq1[TARGET],
    rq1_splits,
    "spatial",
)
rq1_cv = pd.DataFrame([linear_summary, quadratic_summary])
display(rq1_cv[["model", "mae_mean", "mae_std", "rmse_mean", "r2_mean"]].round(3))

if top_coding_detected:
    uncapped = rq1.loc[~np.isclose(rq1[TARGET], target_maximum, rtol=0, atol=1e-9)]
    uncapped_spearman = stats.spearmanr(uncapped["median_income"], uncapped[TARGET])
    display(pd.DataFrame([
        {"analysis": "full recorded target", "spearman": spearman.statistic, "n": len(rq1)},
        {"analysis": "secondary uncapped subset", "spearman": uncapped_spearman.statistic, "n": len(uncapped)},
    ]).round(4))

### RQ1 interpretation

The null hypothesis is zero area-level association between median income and recorded housing value. Pearson emphasizes linear association; Spearman emphasizes monotonic association. The linear coefficient, confidence interval, spatially validated nonlinear comparison, and top-coding sensitivity should be interpreted as observational evidence, not causation.

## 12. Research Question 2 — Incremental geographic predictive information

In [ ]:
rq2_feature_sets = {
    "Model A: income only": ["median_income"],
    "Model B: income plus non-geographic housing": [
        "median_income",
        "housing_median_age",
        "total_rooms",
        "total_bedrooms",
        "population",
        "households",
        *ENGINEERED_FEATURES,
    ],
    "Model C: Model B plus coordinates": [
        "median_income",
        "housing_median_age",
        "total_rooms",
        "total_bedrooms",
        "population",
        "households",
        *ENGINEERED_FEATURES,
        "longitude",
        "latitude",
    ],
}

alpha_grid = {"model__alpha": [1.0, 10.0] if FAST_MODE else [0.1, 1.0, 10.0, 100.0]}
rq2_rows = []

for name, features in rq2_feature_sets.items():
    for strategy, splits, groups in [
        ("spatial", SPATIAL_SPLITS, model_groups),
        ("random", RANDOM_SPLITS, None),
    ]:
        _, summary, _ = nested_evaluate(
            name,
            standard_pipeline(Ridge()),
            alpha_grid,
            X_all[features],
            y_all,
            splits,
            strategy,
            groups,
        )
        rq2_rows.append(summary)

model_c_features = rq2_feature_sets["Model C: Model B plus coordinates"]
geo_features = ["longitude", "latitude"]
other_features = [c for c in model_c_features if c not in geo_features]

geo_processor = ColumnTransformer([
    (
        "geo",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("poly", PolynomialFeatures(degree=2, include_bias=False)),
            ("scale", StandardScaler()),
        ]),
        geo_features,
    ),
    (
        "other",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]),
        other_features,
    ),
])
model_d = Pipeline([("preprocess", geo_processor), ("model", Ridge())])

for strategy, splits, groups in [
    ("spatial", SPATIAL_SPLITS, model_groups),
    ("random", RANDOM_SPLITS, None),
]:
    _, summary, _ = nested_evaluate(
        "Model D: nonlinear geographic terms",
        model_d,
        alpha_grid,
        X_all[model_c_features],
        y_all,
        splits,
        strategy,
        groups,
    )
    rq2_rows.append(summary)

rq2_comparison = pd.DataFrame(rq2_rows)
display(
    rq2_comparison[
        ["model", "validation_strategy", "mae_mean", "mae_std", "rmse_mean", "r2_mean"]
    ].sort_values(["validation_strategy", "mae_mean"]).round(3)
)
rq2_comparison.to_csv(TABLE_DIRECTORY / "rq2_model_comparison.csv", index=False)

**RQ2 interpretation:** The change from Model B to Model C measures incremental predictive information from location after adjustment for income and non-geographic housing characteristics. Model D evaluates nonlinear geographic structure. Spatial results are primary and do not represent a causal effect of location.

## 13. Research Question 3 — Density and crowding proxies

In [ ]:
rq3_proxies = [
    "population_per_household",
    "rooms_per_household",
    "bedrooms_per_household",
    "bedroom_to_room_ratio",
]
rq3_controls = [
    "median_income",
    "housing_median_age",
    "longitude",
    "latitude",
    "total_rooms",
    "population",
    "households",
]
rq3_data = engineered_data[rq3_proxies + rq3_controls + [TARGET]].replace(
    [np.inf, -np.inf], np.nan
).dropna()

if len(rq3_data) < 100:
    raise ValueError("Insufficient complete observations for RQ3.")

X_rq3 = rq3_data[rq3_proxies + rq3_controls].astype(float)
y_rq3 = rq3_data[TARGET].astype(float)
X_std = (X_rq3 - X_rq3.mean()) / X_rq3.std(ddof=0).replace(0, np.nan)
y_std = (y_rq3 - y_rq3.mean()) / y_rq3.std(ddof=0)

if STATSMODELS_AVAILABLE:
    import statsmodels.api as sm
    fit = sm.OLS(y_std, sm.add_constant(X_std), missing="drop").fit(cov_type="HC3")
    confidence = fit.conf_int()
    rq3_results = pd.DataFrame([
        {
            "feature": feature,
            "standardized_coefficient": fit.params[feature],
            "ci_95_low": confidence.loc[feature, 0],
            "ci_95_high": confidence.loc[feature, 1],
            "p_value": fit.pvalues[feature],
        }
        for feature in rq3_proxies
    ])
else:
    iterations = 100 if FAST_MODE else 500
    base = LinearRegression().fit(X_std, y_std)
    base_map = dict(zip(X_std.columns, base.coef_))
    bootstrap = {feature: [] for feature in rq3_proxies}
    rng = np.random.default_rng(RANDOM_SEED)
    for _ in range(iterations):
        positions = rng.integers(0, len(X_std), len(X_std))
        fitted = LinearRegression().fit(X_std.iloc[positions], y_std.iloc[positions])
        coef_map = dict(zip(X_std.columns, fitted.coef_))
        for feature in rq3_proxies:
            bootstrap[feature].append(coef_map[feature])
    rq3_results = pd.DataFrame([
        {
            "feature": feature,
            "standardized_coefficient": base_map[feature],
            "ci_95_low": np.percentile(bootstrap[feature], 2.5),
            "ci_95_high": np.percentile(bootstrap[feature], 97.5),
            "p_value": np.nan,
        }
        for feature in rq3_proxies
    ])

display(rq3_results.round(5))
print(f"Complete area-level observations used: {len(rq3_data):,}")
print(f"Predictor correlation condition number: {np.linalg.cond(X_rq3.corr().fillna(0).to_numpy()):.2f}")

### RQ3 interpretation

For each proxy, the null hypothesis is a zero adjusted standardized association. These variables are area-level proxies, not direct measures of individual crowding. Multicollinearity can make individual coefficients unstable, and significance does not establish practical importance or causation.

## 14. Research Question 4 — Housing-stock age

In [ ]:
rq4_features = [
    "housing_median_age",
    "median_income",
    "longitude",
    "latitude",
    "total_rooms",
    "total_bedrooms",
    "population",
    "households",
]
rq4_X = X_all[rq4_features]

linear_age_model = standard_pipeline(Ridge(alpha=10))
age_processor = ColumnTransformer([
    (
        "age",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("poly", PolynomialFeatures(degree=2, include_bias=False)),
            ("scale", StandardScaler()),
        ]),
        ["housing_median_age"],
    ),
    (
        "other",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]),
        [f for f in rq4_features if f != "housing_median_age"],
    ),
])
nonlinear_age_model = Pipeline([("preprocess", age_processor), ("model", Ridge(alpha=10))])

_, age_linear_summary = fixed_evaluate(
    "Linear age effect", linear_age_model, rq4_X, y_all, SPATIAL_SPLITS, "spatial"
)
_, age_nonlinear_summary = fixed_evaluate(
    "Quadratic age effect", nonlinear_age_model, rq4_X, y_all, SPATIAL_SPLITS, "spatial"
)
rq4_comparison = pd.DataFrame([age_linear_summary, age_nonlinear_summary])
display(rq4_comparison[["model", "mae_mean", "mae_std", "rmse_mean", "r2_mean"]].round(3))

rq4_complete = engineered_data[rq4_features + [TARGET]].replace([np.inf, -np.inf], np.nan).dropna()
age_grid = np.linspace(
    rq4_complete["housing_median_age"].quantile(0.02),
    rq4_complete["housing_median_age"].quantile(0.98),
    80,
)

if STATSMODELS_AVAILABLE:
    import statsmodels.api as sm
    design = rq4_complete[rq4_features].copy()
    design["age_squared"] = design["housing_median_age"] ** 2
    fit = sm.OLS(rq4_complete[TARGET], sm.add_constant(design)).fit(cov_type="HC3")

    reference = pd.DataFrame({
        feature: np.repeat(float(rq4_complete[feature].median()), len(age_grid))
        for feature in rq4_features
    })
    reference["housing_median_age"] = age_grid
    reference["age_squared"] = age_grid ** 2
    prediction = fit.get_prediction(sm.add_constant(reference, has_constant="add")).summary_frame()
    effect = prediction["mean"].to_numpy()
    low = prediction["mean_ci_lower"].to_numpy()
    high = prediction["mean_ci_upper"].to_numpy()
else:
    fitted = clone(nonlinear_age_model).fit(rq4_complete[rq4_features], rq4_complete[TARGET])
    reference = pd.DataFrame({
        feature: np.repeat(float(rq4_complete[feature].median()), len(age_grid))
        for feature in rq4_features
    })
    reference["housing_median_age"] = age_grid
    effect = fitted.predict(reference)
    low = np.full_like(effect, np.nan)
    high = np.full_like(effect, np.nan)

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.plot(age_grid, effect, label="Adjusted predicted recorded value")
if np.isfinite(low).any():
    ax.fill_between(age_grid, low, high, alpha=0.2, label="95% confidence interval")
ax.set_title("Adjusted relationship between housing-stock age and recorded value")
ax.set_xlabel("Housing median age")
ax.set_ylabel("Adjusted recorded median housing value")
ax.legend()
ax.grid(alpha=0.2)
save_figure(fig, "06_adjusted_age_effect")
plt.show()

**RQ4 interpretation:** Spatial cross-validation tests whether nonlinear age specification improves prediction. The adjusted curve is an association summary and does not show that housing age causes value differences.

## 15. Research Question 5 — Comparative predictive modeling

In [ ]:
model_specs = {
    "Mean baseline": (
        Pipeline([("model", DummyRegressor(strategy="mean"))]),
        {},
    ),
    "Ridge": (
        standard_pipeline(Ridge()),
        {"model__alpha": [1.0, 10.0] if FAST_MODE else [0.1, 1.0, 10.0, 100.0]},
    ),
    "Extra Trees": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", ExtraTreesRegressor(
                n_estimators=120 if FAST_MODE else 300,
                max_features=0.8,
                min_samples_leaf=2,
                random_state=RANDOM_SEED,
                n_jobs=-1,
            )),
        ]),
        {} if FAST_MODE else {
            "model__max_depth": [None, 20],
            "model__min_samples_leaf": [1, 4],
        },
    ),
    "HistGradientBoosting": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", HistGradientBoostingRegressor(
                learning_rate=0.08,
                max_leaf_nodes=31,
                l2_regularization=1.0,
                max_iter=160 if FAST_MODE else 300,
                early_stopping=True,
                random_state=RANDOM_SEED,
            )),
        ]),
        {} if FAST_MODE else {
            "model__learning_rate": [0.05, 0.1],
            "model__max_leaf_nodes": [15, 31],
        },
    ),
}

fold_frames = []
summary_rows = []
selected_parameter_log = {}

for model_name, (pipeline, grid) in model_specs.items():
    for strategy, splits, groups in [
        ("spatial", SPATIAL_SPLITS, model_groups),
        ("random", RANDOM_SPLITS, None),
    ]:
        folds, summary, selected = nested_evaluate(
            model_name,
            pipeline,
            grid,
            X_all,
            y_all,
            splits,
            strategy,
            groups,
        )
        fold_frames.append(folds)
        summary_rows.append(summary)
        selected_parameter_log[(model_name, strategy)] = selected

model_fold_metrics = pd.concat(fold_frames, ignore_index=True)
model_comparison = pd.DataFrame(summary_rows)

display(
    model_comparison[
        [
            "model",
            "validation_strategy",
            "mae_mean",
            "mae_std",
            "mae_ci_low",
            "mae_ci_high",
            "rmse_mean",
            "r2_mean",
            "median_ae_mean",
        ]
    ].sort_values(["validation_strategy", "mae_mean"]).round(3)
)

model_comparison.to_csv(
    TABLE_DIRECTORY / "model_comparison_spatial_random.csv",
    index=False,
)

**RQ5 interpretation:** MAE is primary, RMSE is penalty-sensitive, and R² is descriptive. Spatial validation is the primary generalization estimate. Improvement over the mean baseline indicates predictive utility, not causal explanation.

## 16. Random-versus-spatial validation comparison

In [ ]:
validation_comparison = model_comparison.pivot(
    index="model",
    columns="validation_strategy",
    values="mae_mean",
)
if {"random", "spatial"}.issubset(validation_comparison.columns):
    validation_comparison["random_minus_spatial_mae"] = (
        validation_comparison["random"] - validation_comparison["spatial"]
    )
    validation_comparison["random_mae_advantage_percent"] = (
        100
        * (validation_comparison["spatial"] - validation_comparison["random"])
        / validation_comparison["spatial"]
    )
display(validation_comparison.round(3))

**Interpretation:** Lower error under random validation suggests optimism caused by geographic similarity between training and validation observations.

## 17. Model interpretation on a held-out spatial fold

In [ ]:
best_nonbaseline = model_comparison[
    (model_comparison["validation_strategy"] == "spatial")
    & (model_comparison["model"] != "Mean baseline")
].sort_values("mae_mean").iloc[0]

best_model_name = best_nonbaseline["model"]
best_pipeline, best_grid = model_specs[best_model_name]

train_idx, test_idx = SPATIAL_SPLITS[0]
X_train, X_test = X_all.iloc[train_idx], X_all.iloc[test_idx]
y_train, y_test = y_all.iloc[train_idx], y_all.iloc[test_idx]
training_groups = model_groups.iloc[train_idx]

if best_grid:
    inner = GroupKFold(n_splits=min(SPATIAL_INNER_FOLDS, int(training_groups.nunique())))
    search = GridSearchCV(
        clone(best_pipeline),
        best_grid,
        scoring="neg_mean_absolute_error",
        cv=inner,
        n_jobs=-1,
        error_score="raise",
    )
    search.fit(X_train, y_train, groups=training_groups)
    interpretation_model = search.best_estimator_
else:
    interpretation_model = clone(best_pipeline).fit(X_train, y_train)

importance = permutation_importance(
    interpretation_model,
    X_test,
    y_test,
    scoring="neg_mean_absolute_error",
    n_repeats=5 if FAST_MODE else 15,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

importance_table = pd.DataFrame({
    "feature": MODEL_FEATURES,
    "mae_importance_mean": importance.importances_mean,
    "mae_importance_std": importance.importances_std,
}).sort_values("mae_importance_mean", ascending=False)

display(importance_table.head(12).round(4))

plot_data = importance_table.head(10).sort_values("mae_importance_mean")
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.barh(
    plot_data["feature"],
    plot_data["mae_importance_mean"],
    xerr=plot_data["mae_importance_std"],
)
ax.set_title(f"Held-out permutation importance: {best_model_name}")
ax.set_xlabel("Increase in MAE after permutation")
ax.set_ylabel("Feature")
ax.grid(axis="x", alpha=0.2)
save_figure(fig, "07_permutation_importance")
plt.show()

pd_features = ["median_income", "housing_median_age", "rooms_per_household"]
pd_sample = X_test.sample(min(3000, len(X_test)), random_state=RANDOM_SEED)

fig, axes = plt.subplots(1, len(pd_features), figsize=(15, 4))
PartialDependenceDisplay.from_estimator(
    interpretation_model,
    pd_sample,
    features=pd_features,
    ax=axes,
    kind="average",
)
fig.suptitle(f"Aggregate partial dependence: {best_model_name}")
save_figure(fig, "08_partial_dependence")
plt.show()

print(f"Best primary spatial non-baseline model: {best_model_name}")

**Interpretation:** Permutation importance and partial dependence describe predictive behavior on held-out data. Correlated predictors can divide or obscure importance, and predictive importance does not imply causality.

## 18. Sensitivity and robustness analyses

In [ ]:
sensitivity_rows = []
ridge_fixed = standard_pipeline(Ridge(alpha=10))

without_ratios = [f for f in MODEL_FEATURES if f not in ENGINEERED_FEATURES]
for label, features in [
    ("Ridge with engineered ratios", MODEL_FEATURES),
    ("Ridge without engineered ratios", without_ratios),
]:
    _, summary = fixed_evaluate(
        label,
        ridge_fixed,
        X_all[features],
        y_all,
        SPATIAL_SPLITS,
        "spatial",
    )
    sensitivity_rows.append(summary)

if top_coding_detected:
    uncapped_mask = ~np.isclose(y_all, target_maximum, rtol=0, atol=1e-9)
    X_uncapped = X_all.loc[uncapped_mask]
    y_uncapped = y_all.loc[uncapped_mask]
    groups_uncapped = model_groups.loc[uncapped_mask]
    uncapped_splits = list(
        GroupKFold(
            n_splits=min(SPATIAL_OUTER_FOLDS, int(groups_uncapped.nunique()))
        ).split(X_uncapped, y_uncapped, groups=groups_uncapped)
    )
    _, summary = fixed_evaluate(
        "Ridge secondary analysis excluding maximum-recorded target",
        ridge_fixed,
        X_uncapped,
        y_uncapped,
        uncapped_splits,
        "spatial",
    )
    sensitivity_rows.append(summary)

sensitivity_summary = pd.DataFrame(sensitivity_rows)
display(
    sensitivity_summary[
        ["model", "mae_mean", "mae_std", "rmse_mean", "r2_mean"]
    ].round(3)
)
sensitivity_summary.to_csv(
    TABLE_DIRECTORY / "sensitivity_analysis.csv",
    index=False,
)

**Interpretation:** The full preserved dataset remains primary. Secondary analyses evaluate whether conclusions depend on engineered ratios or the recorded target maximum.

## 19. Preprocessing and exclusion decision record

In [ ]:
decision_log = pd.DataFrame([
    {
        "decision": "Preserve original DataFrame",
        "implementation": "All analysis uses an in-memory copy; the source is unchanged.",
        "affected_count": 0,
        "analysis_role": "primary",
    },
    {
        "decision": "Missing or infinite target",
        "implementation": "Excluded from modeling only.",
        "affected_count": excluded_target_rows,
        "analysis_role": "primary",
    },
    {
        "decision": "Invalid ratio denominator",
        "implementation": "Result set to missing; imputation occurs inside training folds.",
        "affected_count": int(feature_engineering_summary["resulting_missing_count"].max()),
        "analysis_role": "primary",
    },
    {
        "decision": "Statistical outliers",
        "implementation": "No automatic deletion, capping, or winsorization.",
        "affected_count": 0,
        "analysis_role": "primary",
    },
    {
        "decision": "Maximum recorded target",
        "implementation": "Excluded only in a secondary top-coding sensitivity analysis.",
        "affected_count": count_at_maximum if top_coding_detected else 0,
        "analysis_role": "sensitivity",
    },
])
display(decision_log)
decision_log.to_csv(TABLE_DIRECTORY / "preprocessing_decisions.csv", index=False)

## 20. Concise answers to the five research questions

In [ ]:
def describe_strength(value):
    magnitude = abs(value)
    if magnitude < 0.1:
        strength = "weak"
    elif magnitude < 0.3:
        strength = "modest"
    else:
        strength = "substantial"
    direction = "positive" if value > 0 else "negative" if value < 0 else "near-zero"
    return f"{strength} {direction}"

spearman_value = float(
    correlation_results.loc[correlation_results["method"] == "Spearman", "estimate"].iloc[0]
)

rq2_spatial = rq2_comparison[
    rq2_comparison["validation_strategy"] == "spatial"
].set_index("model")
model_b_mae = rq2_spatial.loc["Model B: income plus non-geographic housing", "mae_mean"]
model_c_mae = rq2_spatial.loc["Model C: Model B plus coordinates", "mae_mean"]
geo_change = 100 * (model_b_mae - model_c_mae) / model_b_mae

strongest_proxy = rq3_results.iloc[
    rq3_results["standardized_coefficient"].abs().argmax()
]

age_change = 100 * (
    age_linear_summary["mae_mean"] - age_nonlinear_summary["mae_mean"]
) / age_linear_summary["mae_mean"]

best_spatial = model_comparison[
    model_comparison["validation_strategy"] == "spatial"
].sort_values("mae_mean").iloc[0]

answers = [
    ("RQ1", f"Median income shows a {describe_strength(spearman_value)} monotonic association with recorded housing value (Spearman ρ={spearman_value:.3f})."),
    ("RQ2", f"Adding coordinates changed spatial-validation MAE by {geo_change:.2f}% relative to the non-geographic adjusted model."),
    ("RQ3", f"The largest absolute adjusted standardized proxy coefficient was for `{strongest_proxy['feature']}` ({strongest_proxy['standardized_coefficient']:.3f})."),
    ("RQ4", f"The nonlinear age specification changed spatial-validation MAE by {age_change:.2f}% relative to the linear specification."),
    ("RQ5", f"The best primary spatial result was {best_spatial['model']} with MAE={best_spatial['mae_mean']:.2f}, RMSE={best_spatial['rmse_mean']:.2f}, and R²={best_spatial['r2_mean']:.3f}."),
]

for label, answer in answers:
    display(Markdown(f"### {label}\n\n{answer}\n\nThese results are observational or predictive, not causal."))

## 21. Limitations

1. Area-level findings may not describe individuals or households.
2. The observational design does not support causal inference.
3. Unmeasured policy, neighborhood, environmental, and market factors may confound associations.
4. Geographic blocking reduces but may not eliminate spatial dependence.
5. Top-coded targets represent recorded values rather than uncensored market values.
6. Engineered ratios are proxies and can be unstable when denominators are small.
7. Bounded tuning grids support reproducibility and runtime but do not guarantee global optimality.
8. Aggregate outputs reduce disclosure risk but do not guarantee anonymity.
9. Figures follow common publication-readiness principles and may require journal-specific adjustment.

## 22. Reproducibility and generated-output summary

In [ ]:
table_files = [
    TABLE_DIRECTORY / "rq2_model_comparison.csv",
    TABLE_DIRECTORY / "model_comparison_spatial_random.csv",
    TABLE_DIRECTORY / "sensitivity_analysis.csv",
    TABLE_DIRECTORY / "preprocessing_decisions.csv",
]
all_output_files = [p for p in [*generated_files, *table_files] if p.exists()]

output_inventory = pd.DataFrame({
    "relative_path": [str(p.relative_to(RUN_DIRECTORY)) for p in all_output_files],
    "type": [
        "figure" if p.parent == FIGURE_DIRECTORY else "aggregate_table"
        for p in all_output_files
    ],
})
display(output_inventory)

reproducibility = pd.DataFrame([{
    "random_seed": RANDOM_SEED,
    "source_mode": source_mode,
    "input_rows": int(len(analysis_data)),
    "input_columns": int(analysis_data.shape[1]),
    "modeling_rows": int(len(model_frame)),
    "spatial_outer_folds": SPATIAL_OUTER_FOLDS,
    "random_outer_folds": RANDOM_OUTER_FOLDS,
    "fast_mode": FAST_MODE,
    "statsmodels_available": STATSMODELS_AVAILABLE,
    "generated_file_count": len(all_output_files),
}])
display(reproducibility)

## 23. Supported and unsupported conclusions

### Supported

- Aggregate descriptions of this local area-level dataset.
- Statistical associations with stated assumptions and uncertainty.
- Out-of-sample predictive comparisons under spatial and random validation.
- Sensitivity findings under documented secondary analyses.
- Evidence that random and spatial validation may yield different estimates.

### Unsupported

- Causal effects of income, geography, density, or housing age.
- Individual-house or household conclusions.
- Predictions of uncensored market value when the target is capped.
- Claims that the dataset or notebook is anonymous or fully privacy-preserving.
- Claims of universal journal-format compliance.